# IMPORTS

In [1]:
import json
import os
import pandas as pd
from pathlib import Path

from tqdm import tqdm
from datasets import Dataset
import re
import random


import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split

from transformers import DataCollatorForLanguageModeling
from transformers import TrainerCallback

===============================================

In [2]:
DATA_PATH = "D:/it_rus_main.hh_it2.json"

def load_json_safely(path, max_records=1000):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        
        if first_char == "[":
            data = json.load(f)
            records = data[:max_records]
        else:
            for i, line in enumerate(f):
                try:
                    obj = json.loads(line)
                    records.append(obj)
                    if i >= max_records:
                        break
                except json.JSONDecodeError:
                    continue
    return records


In [3]:
import re
import random
import pandas as pd
from tqdm import tqdm
from datasets import Dataset

def clean_text(x):
    """Очистка от мусора, списков и дублей"""
    if x is None:
        return ""
    if isinstance(x, list):
        cleaned = [str(i).strip() for i in x if i and str(i).strip().lower() != "none"]
        # убираем дубликаты, сохраняя порядок
        seen, unique = set(), []
        for i in cleaned:
            if i not in seen:
                unique.append(i)
                seen.add(i)
        x = " ".join(unique)
    x = re.sub(r"\s+", " ", str(x))
    x = x.replace("\xa0", " ").strip()
    return x


def make_instruction(rec):
    title = clean_text(rec.get("title", ""))
    employer = clean_text(rec.get("employer", []))
    experience = clean_text(rec.get("experience", ""))
    salary = clean_text(rec.get("salary", []))
    skills_raw = rec.get("key_skills", [])
    if isinstance(skills_raw, list):
        skills_raw = [s for s in skills_raw if s and s.lower() != "none"]
    skills = ", ".join(skills_raw[:5]) if skills_raw else ""

    parts = []
    if employer:
        parts.append(f"Составь описание вакансии для компании {employer}.")
    if title:
        parts.append(f"Должность: {title}.")
    if skills:
        parts.append(f"Ключевые навыки: {skills}.")
    if experience:
        parts.append(f"Опыт работы: {experience}.")
    if salary:
        parts.append(f"Укажи зарплату: {salary}.")
    return " ".join(parts).strip()

In [4]:
def make_output(rec):
    desc = rec.get("description", [])
    if not desc:
        return None

    if isinstance(desc, list):
        desc = " ".join(map(str, desc))
    desc = clean_text(desc)

    strongs = rec.get("strong_fields", [])
    if strongs:
        for sf in strongs:
            if not sf:
                continue
            sf_clean = clean_text(sf)
            desc = re.sub(
                rf'(^|\n|\.\s|\:\s)({re.escape(sf_clean)})(?!\*\*)',
                lambda m: ("\n\n" if not m.group(1).startswith("\n") and m.group(1) != "" else "") + f"**{m.group(2)}:**\n",
                desc
            )


    if len(desc) < 50:
        return None

    salary = clean_text(rec.get("salary", []))
    if salary and salary.lower() not in desc.lower():
        add_templates = [
            f" Зарплата: {salary}.",
            f" Предлагаем зарплату {salary}.",
            f" Вознаграждение: {salary}.",
            f" Уровень дохода: {salary}.",
        ]
        desc = desc.rstrip(".") + random.choice(add_templates)

    return desc


data = load_json_safely(DATA_PATH, max_records=10_000)

before = len(data)
seen_descriptions = set()
unique_data = []
for rec in data:
    desc = rec.get("description")
    if not desc:
        continue
    desc_text = " ".join(map(str, desc)) if isinstance(desc, list) else str(desc)
    desc_text = clean_text(desc_text)
    if desc_text not in seen_descriptions:
        seen_descriptions.add(desc_text)
        unique_data.append(rec)
after = len(unique_data)
removed = before - after
print(f"Удалено полных дублей по description: {removed} из {before}")

rows = []
for r in tqdm(unique_data, desc="Building dataset"):
    instr = make_instruction(r)
    out = make_output(r)
    if instr and out:
        rows.append({"instruction": instr, "output": out})

print(f"Собрано пар после очистки: {len(rows)}")

if rows:
    dataset = Dataset.from_pandas(pd.DataFrame(rows))
    print(dataset)

    for ex in dataset.select(range(min(3, len(dataset)))):
        print("\n=== INSTRUCTION ===")
        print(ex["instruction"])
        print("\n=== OUTPUT ===")
        print(ex["output"][:600], "...")
else:
    print("❗ Нет валидных записей. Проверь наличие поля 'description' в JSON.")

Удалено полных дублей по description: 529 из 10000


Building dataset: 100%|██████████████████████████████████████████████████████████| 9471/9471 [00:07<00:00, 1272.02it/s]


Собрано пар после очистки: 9471
Dataset({
    features: ['instruction', 'output'],
    num_rows: 9471
})

=== INSTRUCTION ===
Составь описание вакансии для компании Сбер для экспертов. Должность: Руководитель направления финансово-экономической оценки продуктов. Опыт работы: 3–6 лет. Укажи зарплату: от 130 000 ₽ за месяц.

=== OUTPUT ===
Отдел экономической оценки ищет специалиста, который будет заниматься финансово-экономической оценкой продуктов и инициатив в Группе Сбер, имеет проактивную жизненную позицию и готов работать в кросс-территориальной команде

**Обязанности:**
 оценка экономической эффективности продуктов и инициатив всех направлений бизнеса Группы; создание методологии и инструментов оценки, продуктовой аналитики; консультирование бизнес-подразделений по вопросам финансово-экономической оценки; согласование продуктовых решений

**Требования:**
 профильное финансовое, экономическое или математическое образование ...

=== INSTRUCTION ===
Составь описание вакансии для комп

===============================================

In [5]:
df = pd.DataFrame(dataset)
train_df, test_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

Train: 8997 | Test: 474


In [6]:
def tokenize_batch(batch):
    text = [
        f"Вопрос: {i}\n\nОтвет: {o}"
        for i, o in zip(batch["instruction"], batch["output"])
    ]
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tokenized = train_dataset.map(tokenize_batch, batched=True, remove_columns=train_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_batch, batched=True, remove_columns=test_dataset.column_names)

print(train_tokenized)
print(test_tokenized)

Map:   0%|          | 0/8997 [00:00<?, ? examples/s]

Map:   0%|          | 0/474 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8997
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 474
})


In [7]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,             
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# === Настройки LoRA ===
lora_config = LoraConfig(
    r=10,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable, total = 0, 0
for _, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f"Обучаемых параметров: {trainable/1e6:.2f} M / Всего: {total/1e6:.2f} M ({100*trainable/total:.2f} %)")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Обучаемых параметров: 0.68 M / Всего: 494.71 M (0.14 %)


In [8]:
#  Callback: генерация вакансии после каждой валидации ===
class VacancyGenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, model, prompt, gen_kwargs=None):
        self.tokenizer = tokenizer
        self.model = model
        self.prompt = prompt
        self.gen_kwargs = gen_kwargs or {
            "max_new_tokens": 300,
            "temperature": 0.8,
            "top_p": 0.9,
            "do_sample": True,
        }

    def on_evaluate(self, args, state, control, **kwargs):
        print("\n" + "=" * 60)
        print(f"Контрольная генерация на шаге {state.global_step}")
        print("=" * 60)
        inputs = self.tokenizer(
            self.prompt, return_tensors="pt"
        ).to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, **self.gen_kwargs)
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(text)
        print("=" * 60 + "\n")


# Базовый промпт для мониторинга ===
control_prompt = "Составь описание вакансии для компании X. Должность: Аналитик данных. Опыт работы: 3–6 лет. Укажи зарплату: от 150 000 ₽ за месяц."

In [9]:
training_args = TrainingArguments(
    output_dir="./qwen_sft_checkpoints",
    num_train_epochs=6,                   
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,        
    eval_strategy="steps",
    eval_steps=25,                       
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    fp16=True,
    save_total_limit=1,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_steps=-1,
)

In [10]:
# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Callback для контрольной генерации
generation_callback = VacancyGenerationCallback(
    tokenizer=tokenizer,
    model=model,
    prompt=control_prompt,
    gen_kwargs={"max_new_tokens": 1000, "temperature": 0.8, "top_p": 0.9}
)


In [11]:
# Промпт для проверки
initial_prompt = (
    "Составь описание вакансии для компании X. "
    "Должность: Аналитик данных. Опыт работы: 3–6 лет. "
    "Укажи зарплату: от 150 000 ₽ за месяц."
)

print("=== Генерация до обучения ===")
inputs = tokenizer(initial_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=550,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,   
        no_repeat_ngram_size=3,   
    )


generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


=== Генерация до обучения ===


C:\Users\BATMAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Составь описание вакансии для компании X. Должность: Аналитик данных. Опыт работы: 3–6 лет. Укажи зарплату: от 150 000 ₽ за месяц. Подписывайся и получи бесплатную святоотворенную консультацию.
Конечно, давайте создадим описательный текст о этой вакансию:

**Описание Вакансий**

Работа с информационными данными, обеспечивая эффективное управление ресурсами предприятия, является одним из ключевых функций аналитиков. Некоторые из наиболее важных аспектов деятельности аналитических специалистов заключаются в их анализе и вычислении значений. На сегодняшний день, по данным исследования, 94% специализирующихся на анализе информации и исследовании подобных вопросов являются аналитиком.

Известно, что опыт работы в области аналитики составляет всего около трех-пяти лет. Для достижения стажа требуются неплохие знания и умение обрабатывать данные. 

Вот уже есть заявки на эту вакантные должности:

1. **Анализ клиентской данных**: Позже мы будем рассказывать больше подробностей о таких должностя

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[generation_callback],
)


metrics = trainer.evaluate()
print("\n=== METRICS ===")
print(metrics)


C:\Users\BATMAN\AppData\Local\Temp\ipykernel_16708\4055841838.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



Контрольная генерация на шаге 0
Составь описание вакансии для компании X. Должность: Аналитик данных. Опыт работы: 3–6 лет. Укажи зарплату: от 150 000 ₽ за месяц. Отметьте наличие опыта работы, связанных с информационными системами и программированием.
Контент:

**Описание**

Вашей компанию X компания стремится к развитию и процветании. Мы предлагаем высокие зарплаты, в то же время поддерживая надежную работу и динамическое развитие. Ваши обязанности включают анализ и анализ данных, а также обеспечение эффективной работы над информационными системами.

**Работа**

- Анализ и анализ данных
- Предсказание роста рынка и изменений технологий
- Обеспечение достоверных и актуальных данных
- Информационная безопасность

**Опыт работы**

- 3–6 лет
- Зарплата от 150 000 ₽ за месяц

**Одобрение**

Награждение для победителей конкурса

---

**Пожалуйста, укажите свои данные на сайте или приглашение, чтобы мы могли обработать ваше заявление и создать более точный вариант описания вакансии.**

**И

In [ ]:
train_result = trainer.train(resume_from_checkpoint="./qwen_sft_checkpoints/checkpoint-500")

	logging_steps: 25 (from args) != 50 (from trainer_state.json)
	eval_steps: 25 (from args) != 100 (from trainer_state.json)
	save_steps: 25 (from args) != 100 (from trainer_state.json)
	per_device_train_batch_size: 1 (from args) != 2 (from trainer_state.json)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
C:\Users\BATMAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Model Preparation Time
600,1.893300,1.866102,0.001000



Контрольная генерация на шаге 600
Составь описание вакансии для компании X. Должность: Аналитик данных. Опыт работы: 3–6 лет. Укажи зарплату: от 150 000 ₽ за месяц. Зарплата: от 20 000 ₽ за месяц

---

**Информация о компании:**
 Компания X — это крупная и высокотехнологичная компания, специализирующаяся на разработке и производстве программного обеспечения. Согласно официальным данным, компания занимает первое место по числу заказов в России в сфере автоматизации (открытие первой российской платформы управления рисками). История нашей компании продолжается более двух десятилетий. Мы стремимся к достижению высочайших стандартов качества и качества работы наших сотрудников.

**Условия работы::**
 Обязанности: Анализ данных для разработки бизнес-решений; Разработка и внедрение аналитических решений по задачам; Работа с базами данных; Выявление потребностей пользователей в поддержке; Подготовка статистики и анализы для аналитического продукта; Продвижение бизнес-процессов через маркетинг

C:\Users\BATMAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [ ]:
final_metrics = trainer.evaluate()
print("\n=== Финальные метрики ===")
print(final_metrics)

In [ ]:
# output_dir = "./qwen_sft_lora_final"
# trainer.save_model(output_dir)
# tokenizer.save_pretrained(output_dir)